# 🔧 Fase 3 — Do Texto Bruto ao Modelo
## Pipeline Clássico (TF-IDF + ML) vs Transformer (NorBERTo)

---
**Roteiro-Desafio-NLP · Fatec DSM 6º Semestre · 2026**  
**Grupo:** ___________________________  **Data:** ___/___/______

---

### 🎯 Objetivo desta fase
Construir um **pipeline NLP completo** — desde o texto bruto até a predição — usando primeiro a abordagem **clássica** (pré-processamento + TF-IDF + scikit-learn) e depois substituindo pelo **NorBERTo**. A comparação revela o impacto real dos transformers.

### 📚 O que você vai aprender
- Pré-processamento de texto em PT-BR (tokenização, stopwords, lematização)
- Representação TF-IDF e suas limitações
- Como o NorBERTo supera as limitações do TF-IDF
- Análise de erros: onde cada abordagem falha

---


## 📦 Passo 1 — Instalação

In [ ]:
!pip install transformers datasets scikit-learn nltk spacy torch matplotlib seaborn -q
!python -m spacy download pt_core_news_sm -q
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
print("✅ Pronto!")


## 🧹 Passo 2 — Pré-processamento Clássico de Texto
O pré-processamento clássico envolve limpar e normalizar o texto antes de qualquer modelo.


In [ ]:
import re
import string
import nltk
from nltk.corpus import stopwords
import spacy

nlp_spacy = spacy.load("pt_core_news_sm")
STOPWORDS_PT = set(stopwords.words("portuguese"))

def preprocessar_classico(texto):
    """
    Pipeline clássico de pré-processamento para PT-BR:
    1. Minúsculas
    2. Remove pontuação e números
    3. Remove stopwords
    4. Lematização
    """
    # 1. Minúsculas
    texto = texto.lower()
    # 2. Remove pontuação e números
    texto = re.sub(r'[^a-záéíóúâêôãõàüç\s]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    # 3. Tokenização e remoção de stopwords + lematização
    doc = nlp_spacy(texto)
    tokens = [token.lemma_ for token in doc
              if token.text not in STOPWORDS_PT and len(token.text) > 2]
    return " ".join(tokens)

# Demonstração
exemplos = [
    "O Banco Central do Brasil aumentou a taxa de juros em 0,25 ponto percentual.",
    "Meu coração está cheio de amor e felicidade hoje!",
    "Python é uma linguagem de programação muito poderosa e versátil.",
]
print("="*65)
print("COMPARAÇÃO: Texto Original vs Pré-processado")
print("="*65)
for ex in exemplos:
    proc = preprocessar_classico(ex)
    print(f"\nOriginal  : {ex}")
    print(f"Processado: {proc}")
print("="*65)


## 🔢 Passo 3 — Representação TF-IDF
O TF-IDF (Term Frequency-Inverse Document Frequency) converte textos em vetores numéricos, dando mais peso a palavras raras e informativas.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report
from datasets import load_dataset
import numpy as np

# Carrega dataset
print("⏳ Carregando dataset ASSIN2...")
ds = load_dataset("assin2")
train_raw = ds["train"]
test_raw  = ds["test"]

# Prepara textos (combina premissa + hipótese para entailment)
def preparar_par(ex):
    return ex["premise"] + " [SEP] " + ex["hypothesis"]

X_train_raw = [preparar_par(ex) for ex in train_raw]
X_test_raw  = [preparar_par(ex) for ex in test_raw]
y_train = train_raw["entailment_judgment"]
y_test  = test_raw["entailment_judgment"]

print(f"✅ Dataset: {len(X_train_raw)} treino, {len(X_test_raw)} teste")
print(f"   Classes: {set(y_train)}")

# Pré-processamento clássico
print("\n⏳ Pré-processando textos...")
X_train_proc = [preprocessar_classico(t) for t in X_train_raw]
X_test_proc  = [preprocessar_classico(t) for t in X_test_raw]
print("✅ Pré-processamento concluído!")

# TF-IDF
print("\n⏳ Aplicando TF-IDF...")
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X_train_tfidf = tfidf.fit_transform(X_train_proc)
X_test_tfidf  = tfidf.transform(X_test_proc)
print(f"✅ Matriz TF-IDF: {X_train_tfidf.shape}")

# Treinamento — Regressão Logística
print("\n⏳ Treinando Regressão Logística...")
clf = LogisticRegression(max_iter=500, random_state=42)
clf.fit(X_train_tfidf, y_train)
preds_tfidf = clf.predict(X_test_tfidf)
f1_tfidf = f1_score(y_test, preds_tfidf, average="macro")
print(f"\n📊 RESULTADO — TF-IDF + Regressão Logística")
print(f"   F1-Score (macro): {f1_tfidf:.4f}")
print(classification_report(y_test, preds_tfidf))


## 🚀 Passo 4 — Substituindo pelo NorBERTo
Agora vamos repetir a tarefa usando o NorBERTo. Sem pré-processamento manual — o modelo cuida de tudo!


In [ ]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
import torch

MODEL_NAME = "Itau-Unibanco/NorBERTo-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

label2id = {l: i for i, l in enumerate(sorted(set(y_train)))}
id2label  = {v: k for k, v in label2id.items()}
NUM_CLASSES = len(label2id)

class PairDataset(torch.utils.data.Dataset):
    def __init__(self, premissas, hipoteses, rotulos):
        self.enc = tokenizer(premissas, hipoteses, truncation=True,
                             padding=True, max_length=128, return_tensors="pt")
        self.labels = torch.tensor([label2id[r] for r in rotulos])
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.labels[i]
        return item

train_prem = train_raw["premise"]
train_hip  = train_raw["hypothesis"]
test_prem  = test_raw["premise"]
test_hip   = test_raw["hypothesis"]

train_ds = PairDataset(train_prem, train_hip, y_train)
test_ds  = PairDataset(test_prem,  test_hip,  y_test)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_CLASSES, ignore_mismatched_sizes=True)

args = TrainingArguments(
    output_dir="./norberto_fase3",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    evaluation_strategy="epoch",
    save_strategy="no",
    report_to="none",
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return {"f1_macro": f1_score(labels, preds, average="macro")}

trainer = Trainer(model=model, args=args,
                  train_dataset=train_ds, eval_dataset=test_ds,
                  compute_metrics=compute_metrics)

print("⏳ Treinando NorBERTo...")
trainer.train()
results = trainer.evaluate()
f1_norberto = results["eval_f1_macro"]
print(f"\n📊 RESULTADO — NorBERTo-base")
print(f"   F1-Score (macro): {f1_norberto:.4f}")


## 📊 Passo 5 — Comparação e Análise de Erros

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Gráfico comparativo
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Pipeline Clássico vs NorBERTo — Fase 3", fontsize=14, fontweight='bold')

modelos = ["TF-IDF +\nLog. Reg.", "NorBERTo-base"]
f1s = [f1_tfidf, f1_norberto]
cores = ["#94A3B8", "#0D9488"]

axes[0].bar(modelos, f1s, color=cores)
axes[0].set_title("F1-Score (Macro)")
axes[0].set_ylim(0, 1)
for i, v in enumerate(f1s):
    axes[0].text(i, v+0.01, f"{v:.4f}", ha='center', fontweight='bold', fontsize=12)

# Matriz de confusão — TF-IDF
cm_tfidf = confusion_matrix(y_test, preds_tfidf)
sns.heatmap(cm_tfidf, annot=True, fmt='d', ax=axes[1],
            xticklabels=id2label.values(), yticklabels=id2label.values(),
            cmap="Blues")
axes[1].set_title("Conf. Matrix — TF-IDF")
axes[1].set_xlabel("Predito"); axes[1].set_ylabel("Real")

# Análise de diferença de F1 por classe
delta = f1_norberto - f1_tfidf
axes[2].bar(["Melhoria\nNorBERTo vs TF-IDF"], [delta],
            color="#10B981" if delta > 0 else "#EF4444")
axes[2].set_title("Ganho de F1 (NorBERTo - TF-IDF)")
axes[2].axhline(0, color='black', linewidth=0.8)
axes[2].text(0, delta + 0.005 * (1 if delta>0 else -1),
             f"{delta:+.4f}", ha='center', fontweight='bold', fontsize=14)

plt.tight_layout()
plt.savefig("pipeline_fase3.png", dpi=150, bbox_inches='tight')
plt.show()

print("\n💡 DISCUSSÃO:")
print(f"   TF-IDF + LR : {f1_tfidf:.4f}")
print(f"   NorBERTo    : {f1_norberto:.4f}")
print(f"   Ganho       : {f1_norberto - f1_tfidf:+.4f}")
print("\n   O NorBERTo não precisa de pré-processamento manual,")
print("   pois o modelo aprendeu representações contextuais ricas do português.")


## 📝 Avaliação — Fase 3

**Q1.** O que significa TF-IDF?  
( ) Token Frequency — Inverse Document Filter  
( ) Term Frequency — Inverse Document Frequency  
( ) Text Feature — Indexed Document File  
( ) Token Finder — Indexed Dictionary Format

**Q2.** Por que removemos stopwords no pré-processamento clássico?  
( ) Porque o modelo não consegue processá-las  
( ) São palavras muito frequentes que carregam pouca informação semântica  
( ) Para reduzir o vocabulário abaixo de 1000 palavras  
( ) Porque o TF-IDF não suporta palavras com acento

**Q3.** O que é lematização?  
( ) Remover todas as vogais de uma palavra  
( ) Converter palavras para sua forma raiz/base (ex: "correu" → "correr")  
( ) Dividir palavras compostas em subpalavras  
( ) Converter texto para minúsculas

**Q4.** Qual a principal limitação do TF-IDF em relação ao NorBERTo?  
( ) O TF-IDF é muito mais lento de calcular  
( ) O TF-IDF não consegue processar texto em português  
( ) O TF-IDF ignora o contexto e o significado das palavras  
( ) O TF-IDF usa muita memória RAM

**Q5.** Por que o NorBERTo não precisa de pré-processamento manual (stopwords, lematização)?  
( ) Porque ele ignora palavras irrelevantes automaticamente  
( ) Porque seus embeddings contextuais capturam o significado sem essas etapas  
( ) Porque ele foi treinado sem stopwords no corpus  
( ) Porque o tokenizador remove stopwords antes da tokenização

**Q6.** O que é ngram_range=(1,2) no TfidfVectorizer?  
( ) Limita o vocabulário a palavras com 1 ou 2 caracteres  
( ) Considera unigramas (1 palavra) e bigramas (2 palavras consecutivas)  
( ) Usa apenas as 2 palavras mais frequentes do corpus  
( ) Define o número mínimo e máximo de documentos por classe

**Q7.** No experimento de inferência textual (ASSIN2), o que o modelo recebe como entrada?  
( ) Apenas a premissa  
( ) Apenas a hipótese  
( ) A premissa e a hipótese concatenadas  
( ) Um vetor de features manuais da premissa

**Q8.** O que é overfitting em modelos de classificação de texto?  
( ) O modelo aprende muito devagar durante o treino  
( ) O modelo memoriza o treino mas generaliza mal para dados novos  
( ) O modelo usa muita memória durante a inferência  
( ) O modelo tem F1 baixo tanto no treino quanto no teste

**Q9.** A lematização usa qual biblioteca Python nesta fase?  
( ) NLTK  ( ) spaCy  ( ) Gensim  ( ) HuggingFace Transformers

**Q10.** O que o parâmetro max_features=10000 no TfidfVectorizer faz?  
( ) Limita o texto a 10.000 caracteres  
( ) Mantém apenas as 10.000 palavras mais frequentes no vocabulário TF-IDF  
( ) Define o número máximo de documentos no treino  
( ) Limita o número de épocas de treinamento
